# CIFAR-10 STAQ

A notebook for the CIFAR-10 STAQ workflow:

- load config, CLIP, concepts, and data
- load or train Concept-QA
- load or train baseline and STAQ
- optionally compare `baseline` vs `lam=0.4` with fixed-history summary numbers
- probe the sensitive concept set before retraining
- check sample-level sensitive-label sanity
- replay tiny-start cases

In [ ]:
%pip install -e ..

In [1]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

from staq.analysis import (
    evaluate_bundles_on_fixed_histories,
    plot_rollout_comparisons,
    sample_intuition_replays,
)
from staq.config import Cifar10StaqConfig, default_paths
from staq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_concepts,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from staq.data import get_cifar10_datasets, get_cifar10_loaders, get_raw_cifar10_dataset
from staq.models import ConceptNet2
from staq.sensitive_labels import (
    build_cifar10_sensitive_match,
    build_sensitive_labels,
    load_sensitive_labels,
    save_sensitive_labels,
)
from staq.training import HistorySamplingConfig, build_staq_models, fit_concept_qa, fit_staq, seed_everything
from staq.training.concept_qa import load_gpt_answers

In [2]:
repo_root = Path.cwd().resolve()
if not (repo_root / "staq").exists() and (repo_root.parent / "staq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = Cifar10StaqConfig()
device = config.device
seed_everything(config.random_seed)

print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")

repo_root: /home/jupyter/staq
artifacts_root: /home/jupyter/staq/artifacts
device: cuda


In [3]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
concepts = load_concepts(paths.concept_file)
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sensitive_match = build_cifar10_sensitive_match(concepts)
sens_idx = sensitive_match.indices
sensitive_mask = make_sensitive_mask(config.max_queries, sens_idx, device)

train_ds, test_ds = get_cifar10_datasets(transform=preprocess, root=paths.data_root)
train_loader, test_loader = get_cifar10_loaders(
    transform=preprocess,
    root=paths.data_root,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)
raw_test_ds = get_raw_cifar10_dataset(paths.data_root, train=False)

print(f"# concepts: {len(concepts)}")
print(f"# sensitive concepts matched: {len(sensitive_match.matched)}")
print(sensitive_match.matched)
if sensitive_match.missing:
    print(f"# sensitive concepts missing: {len(sensitive_match.missing)}")
    print(sensitive_match.missing)

# concepts: 128
# sensitive concepts matched: 23
['a bridle', 'a cab for the driver', 'a captain', 'a collar', 'a copilot', 'a dashboard', 'a driver', 'a flight attendant', 'a gear shift', 'a halter', 'a hitch', 'a lead rope', 'a leash', 'a passenger', 'a pedal', 'a pilot', 'a reins', 'a rider', 'a rifle', 'a saddle', 'a seatbelt', 'a steering wheel', 'a trailer']


In [4]:
qa_checkpoint = paths.checkpoints_root / "concept_qa_cifar10.pt"
qa_source = qa_checkpoint
if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
elif paths.bootstrap_concept_qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(paths.bootstrap_concept_qa_checkpoint, device=device)
    qa_source = paths.bootstrap_concept_qa_checkpoint
elif paths.gpt_answers_file.exists():
    gpt_answers = load_gpt_answers(paths.gpt_answers_file)
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.SGD(qa_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(qa_optimizer, T_max=200)
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=train_loader,
        eval_loader=test_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=2,
        model_clip=model_clip,
        dictionary=dictionary,
        gpt_answers=gpt_answers,
        clip_device=device,
        train_device=device,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(paths.runs_root / "concept_qa_cifar10_history.json", "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
else:
    raise FileNotFoundError("No local Concept-QA checkpoint, bootstrap checkpoint, or GPT answers file available.")

print(f"Concept-QA ready from: {qa_source}")

Concept-QA ready from: /home/jupyter/staq/artifacts/checkpoints/bootstrap/concept_qa_cifar10_reference.pth


In [5]:
sensitive_labels_dir = paths.sensitive_labels_root

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels(
        loader=train_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (train)",
    )
    s_soft_test, s_hard_test = build_sensitive_labels(
        loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (test)",
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Hard-positive rate (train/test):",
    float(sensitive_label_cache["s_hard_train"].mean()),
    float(sensitive_label_cache["s_hard_test"].mean()),
)

Sensitive labels ready from: cache -> /home/jupyter/staq/artifacts/sensitive_labels/cifar10
{'s_soft_train': (50000,), 's_hard_train': (50000,), 's_soft_test': (10000,), 's_hard_test': (10000,)}
Hard-positive rate (train/test): 0.5468999743461609 0.5491999983787537


In [7]:
def load_or_train_bundle(
    run_name,
    lambda_adv,
    alpha_sens,
    min_history=config.min_history,
    max_history=config.max_history,
    non_sensitive_only=config.non_sensitive_history_only,
    epochs=2,
    learning_rate=config.learning_rate,
    max_train_batches=60 if device.type == "cpu" else None,
    max_test_batches=30 if device.type == "cpu" else None,
    force_retrain=False,
):
    ckpt_path = paths.checkpoints_root / f"{run_name}_best.pt"
    history_path = paths.runs_root / f"{run_name}_history.json"
    if ckpt_path.exists() and not force_retrain:
        return load_run_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)

    actor_checkpoint = str(paths.bootstrap_actor_checkpoint) if paths.bootstrap_actor_checkpoint.exists() else None
    classifier_checkpoint = str(paths.bootstrap_classifier_checkpoint) if paths.bootstrap_classifier_checkpoint.exists() else None

    actor, classifier, s_head = build_staq_models(
        max_queries=config.max_queries,
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
        actor_checkpoint=actor_checkpoint,
        classifier_checkpoint=classifier_checkpoint,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_staq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_adv=lambda_adv,
        alpha_sens=alpha_sens,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_test_batches,
    )
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        metadata={
            "run_name": run_name,
            "lambda_adv": lambda_adv,
            "alpha_sens": alpha_sens,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return load_run_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)


baseline_bundle = load_or_train_bundle("baseline", lambda_adv=0.0, alpha_sens=0.0, epochs=5)
staq_bundle = load_or_train_bundle("lam_0.40", lambda_adv=0.4, alpha_sens=0.0, epochs=5)

print(baseline_bundle["ckpt_path"])
print(staq_bundle["ckpt_path"])

def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )

STAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

/home/jupyter/staq/artifacts/checkpoints/baseline_best.pt
/home/jupyter/staq/artifacts/checkpoints/lam_0.40_best.pt


In [8]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    staq_bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=raw_test_ds.classes,
    num_cases=6,
    pool_size=400 if device.type == "cpu" else 1500,
    num_trials=3,
    min_history=0,
    max_history=1,
    history_mode="non_sensitive",
    prefer_baseline_sensitive=True,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=paths.figures_root / "cifar10_intuition_replay_examples.png",
    title_prefix="tiny-start replay",
)

print(f"Saved tiny-start replay figure: {intuition_fig}")

Sampling intuition replays:   0%|          | 0/1500 [00:00<?, ?it/s]

Saved tiny-start replay figure: /home/jupyter/staq/artifacts/figures/cifar10_intuition_replay_examples.png


## Optional Comparison

For CIFAR-10, keep the aggregate comparison minimal.
Use this section only to compare `baseline` vs `lam=0.4` with fixed-history summary numbers.
We do not rely on sweep charts here because the aggregate story is weak and noisy.
If you later want a wider hyperparameter study, do it separately from the main CIFAR-10 notebook path.

In [9]:
def make_comparison_run_name(comparison_name, lambda_adv, alpha_sens):
    if lambda_adv == 0.0 and alpha_sens == 0.0:
        return f"{comparison_name}_baseline"
    run_name = f"{comparison_name}_lam_{lambda_adv:.2f}"
    if alpha_sens != 0.0:
        run_name += f"_alpha_{alpha_sens:.2f}"
    return run_name


def make_comparison_label(lambda_adv, alpha_sens):
    if lambda_adv == 0.0 and alpha_sens == 0.0:
        return "baseline"
    label = f"lam={lambda_adv:.1f}"
    if alpha_sens != 0.0:
        label += f", alpha={alpha_sens:.1f}"
    return label


comparison_name = "cifar10_baseline_vs_lam_0.4_e5"
comparison_lambda_values = [0.0, 0.4]
comparison_alpha_values = [0.0]

comparison_epochs = 5
comparison_learning_rate = config.learning_rate
comparison_min_history = config.min_history
comparison_max_history = config.max_history
comparison_non_sensitive_only = config.non_sensitive_history_only

# Keep the comparison notebook-friendly on CPU. Set these to None on CUDA for full runs.
comparison_max_train_batches = 60 if device.type == "cpu" else None
comparison_max_test_batches = 30 if device.type == "cpu" else None

# Use a fresh comparison_name for a new run. Set this to True only when reusing the same names.
comparison_force_retrain = False

comparison_num_trials = 8
comparison_history_mode = "non_sensitive"
comparison_max_samples = 2000 if device.type == "cpu" else None

comparison_bundles = {}

for alpha_sens in comparison_alpha_values:
    for lambda_adv in comparison_lambda_values:
        run_name = make_comparison_run_name(comparison_name, lambda_adv, alpha_sens)
        label = make_comparison_label(lambda_adv, alpha_sens)
        print(f"Loading/training {label} -> {run_name}")
        comparison_bundles[label] = load_or_train_bundle(
            run_name=run_name,
            lambda_adv=lambda_adv,
            alpha_sens=alpha_sens,
            min_history=comparison_min_history,
            max_history=comparison_max_history,
            non_sensitive_only=comparison_non_sensitive_only,
            epochs=comparison_epochs,
            learning_rate=comparison_learning_rate,
            max_train_batches=comparison_max_train_batches,
            max_test_batches=comparison_max_test_batches,
            force_retrain=comparison_force_retrain,
        )

Loading/training baseline -> cifar10_baseline_vs_lam_0.4_e5_baseline


STAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Loading/training lam=0.4 -> cifar10_baseline_vs_lam_0.4_e5_lam_0.40


STAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

In [10]:
comparison_fixed_history_rows = evaluate_bundles_on_fixed_histories(
    dataset=test_ds,
    answer_builder=answer_builder,
    bundles_by_name=comparison_bundles,
    sensitive_mask=sensitive_mask,
    min_history=comparison_min_history,
    max_history=comparison_max_history,
    history_mode=comparison_history_mode,
    num_trials=comparison_num_trials,
    max_samples=comparison_max_samples,
    eval_seed=config.random_seed,
    desc="Fixed-history comparison",
)

comparison_fixed_history_path = paths.runs_root / f"{comparison_name}_fixed_history_summary.json"
with open(comparison_fixed_history_path, "w", encoding="utf-8") as handle:
    json.dump(comparison_fixed_history_rows, handle, indent=2)

baseline_row = next(row for row in comparison_fixed_history_rows if row["run_name"] == "baseline")
baseline_acc = baseline_row["mean_acc"]
baseline_sens = baseline_row["mean_sensitive_query_rate"]

comparison_table = [
    {
        "run_name": row["run_name"],
        "lambda_adv": row["lambda_adv"],
        "mean_acc": round(row["mean_acc"], 4),
        "acc_delta_vs_baseline": round(row["mean_acc"] - baseline_acc, 4),
        "std_acc": round(row["std_acc"], 4),
        "mean_sensitive_query_rate": round(row["mean_sensitive_query_rate"], 4),
        "sens_delta_vs_baseline": round(row["mean_sensitive_query_rate"] - baseline_sens, 4),
        "std_sensitive_query_rate": round(row["std_sensitive_query_rate"], 4),
        "mean_confidence": round(row["mean_confidence"], 4),
    }
    for row in comparison_fixed_history_rows
]

comparison_table_path = paths.runs_root / f"{comparison_name}_summary_table.json"
with open(comparison_table_path, "w", encoding="utf-8") as handle:
    json.dump(comparison_table, handle, indent=2)

print(f"Saved fixed-history summary: {comparison_fixed_history_path}")
print(f"Saved comparison table: {comparison_table_path}")

comparison_table

Fixed-history comparison:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved fixed-history summary: /home/jupyter/staq/artifacts/runs/cifar10_baseline_vs_lam_0.4_e5_fixed_history_summary.json
Saved comparison table: /home/jupyter/staq/artifacts/runs/cifar10_baseline_vs_lam_0.4_e5_summary_table.json


[{'run_name': 'baseline',
  'lambda_adv': 0.0,
  'mean_acc': 0.3534,
  'acc_delta_vs_baseline': 0.0,
  'std_acc': 0.0054,
  'mean_sensitive_query_rate': 0.0268,
  'sens_delta_vs_baseline': 0.0,
  'std_sensitive_query_rate': 0.0016,
  'mean_confidence': 0.3907},
 {'run_name': 'lam=0.4',
  'lambda_adv': 0.4,
  'mean_acc': 0.3546,
  'acc_delta_vs_baseline': 0.0012,
  'std_acc': 0.0041,
  'mean_sensitive_query_rate': 0.0302,
  'sens_delta_vs_baseline': 0.0034,
  'std_sensitive_query_rate': 0.0018,
  'mean_confidence': 0.3888}]